# Ler um treino

Abre um treino baixado da Garmin e mostra o que tem nele. Usa duas fontes da pasta `data/garmin/` mais recente:

- **JSON da API**: o resumo do treino (duração, FC média, calorias, efeito do treino, tempo em cada zona).
- **Arquivo FIT**: o arquivo original do relógio, com as medições ao longo do treino e, na musculação, cada série.

Para abrir (da raiz do repositório):

```
uv sync
uv run jupyter lab notebooks/01_ler_treino.ipynb
```

No VS Code, basta escolher o kernel do `.venv` do projeto.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
from garmin_fit_sdk import Decoder, Stream

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
PASTA = max((RAIZ / "data" / "garmin").iterdir())  # coleta mais recente
print("Usando:", PASTA.relative_to(RAIZ))


def ler_json(nome: str):
    return json.loads((PASTA / f"{nome}.json").read_text(encoding="utf-8"))

## 1. Treinos da coleta

In [ ]:
atividades = pd.DataFrame(ler_json("atividades"))
atividades["tipo"] = atividades["activityType"].str["typeKey"]
atividades["duracao_min"] = (atividades["duration"] / 60).round(1)
atividades[["activityId", "startTimeLocal", "activityName", "tipo", "duracao_min", "averageHR", "maxHR", "calories"]]

Só os treinos com arquivo FIT baixado podem ser lidos por completo. Por padrão, o notebook pega o primeiro que encontrar; para ler outro, troque `ATIVIDADE_ID`.

In [ ]:
fits = {int(p.parent.name.split("_")[1]): p for p in PASTA.glob("atividade_*_original/*.fit")}
print("Treinos com FIT:", list(fits))

ATIVIDADE_ID = next(iter(fits))
ARQUIVO_FIT = fits[ATIVIDADE_ID]
atividade = atividades.set_index("activityId").loc[ATIVIDADE_ID]

## 2. Resumo (API)

Alguns campos só existem em certos esportes; os que não vieram aparecem vazios.

In [ ]:
resumo = ler_json(f"atividade_{ATIVIDADE_ID}")["summaryDTO"]
pd.Series(
    {
        "Início": resumo["startTimeLocal"],
        "Duração (min)": round(resumo["duration"] / 60, 1),
        "Calorias": resumo.get("calories"),
        "FC média": resumo.get("averageHR"),
        "FC máxima": resumo.get("maxHR"),
        "FC mínima": resumo.get("minHR"),
        "Efeito aeróbico": resumo.get("trainingEffect"),
        "Efeito anaeróbico": resumo.get("anaerobicTrainingEffect"),
        "Séries ativas": resumo.get("activeSets"),
        "Repetições": resumo.get("totalExerciseReps"),
        "Variação do Body Battery": resumo.get("differenceBodyBattery"),
    },
    name="valor",
)

## 3. Tempo em cada zona de FC

In [ ]:
tempo_zonas = pd.DataFrame(
    {"zona": [f"Z{i}" for i in range(1, 6)], "minutos": [atividade[f"hrTimeInZone_{i}"] / 60 for i in range(1, 6)]}
)
px.bar(tempo_zonas, x="zona", y="minutos", title="Tempo em cada zona de FC", text_auto=".1f")

## 4. Arquivo FIT

O FIT é dividido em tipos de mensagem. As principais:

- `record_mesgs`: medições ao longo do treino (FC, temperatura e, em corrida ou pedal, distância, ritmo e GPS).
- `set_mesgs`: séries de musculação (repetições, peso, exercício, descanso).
- `session_mesgs` e `lap_mesgs`: resumo do treino e das voltas.

Os tipos que aparecem como número são mensagens que a Garmin não documenta publicamente.

In [ ]:
mensagens, erros = Decoder(Stream.from_file(str(ARQUIVO_FIT))).read()
print("Erros de leitura:", erros or "nenhum")
pd.Series({tipo: len(lista) for tipo, lista in mensagens.items()}, name="quantidade").sort_values(ascending=False)

O FIT guarda os horários em UTC. A função abaixo transforma um tipo de mensagem em tabela, passa os horários para o fuso do treino e descarta os campos sem nome.

In [ ]:
FUSO = pd.Timestamp(resumo["startTimeLocal"]) - pd.Timestamp(resumo["startTimeGMT"])


def tabela(tipo: str) -> pd.DataFrame:
    df = pd.DataFrame(mensagens.get(tipo, []))
    df = df[[c for c in df.columns if isinstance(c, str)]]
    for coluna in df.select_dtypes("datetimetz"):
        df[coluna] = df[coluna].dt.tz_localize(None) + FUSO
    return df

## 5. Frequência cardíaca ao longo do treino

O intervalo entre os pontos depende da configuração de gravação do relógio (a cada segundo ou inteligente) e pode ter trechos sem gravação. A célula abaixo mostra quantos pontos vieram e o intervalo típico.

In [ ]:
registros = tabela("record_mesgs")
inicio = registros["timestamp"].iloc[0]
registros["minuto"] = (registros["timestamp"] - inicio).dt.total_seconds() / 60
intervalo = registros["timestamp"].diff().dt.total_seconds().median()
print(f"{len(registros)} pontos, intervalo mediano de {intervalo:.0f} s")
registros.head()

In [ ]:
zonas = ler_json("zonas_fc")[0]
limites = [zonas[f"zone{i}Floor"] for i in range(1, 6)] + [zonas["maxHeartRateUsed"]]
cores = ["#4c9be8", "#5cb85c", "#f0ad4e", "#ec7f3c", "#d9534f"]

fig = px.line(registros, x="minuto", y="heart_rate", title="Frequência cardíaca",
              labels={"minuto": "Minutos", "heart_rate": "FC (bpm)"})
for i in range(5):
    fig.add_hrect(y0=limites[i], y1=limites[i + 1], fillcolor=cores[i], opacity=0.12, line_width=0,
                  annotation_text=f"Z{i + 1}", annotation_position="top left")
fig.update_yaxes(range=[registros["heart_rate"].min() - 5, registros["heart_rate"].max() + 5])
fig

## 6. Séries (musculação)

Cada série ativa é seguida de um descanso. O campo `category` é o palpite do relógio sobre o exercício e vem como lista; aqui ficam só os valores conhecidos, sem repetição. O peso só é real se foi informado no relógio.

In [ ]:
series = tabela("set_mesgs")
if series.empty:
    print("Este treino não tem séries.")
else:
    series["descanso_s"] = series["duration"].shift(-1).where(series["set_type"].shift(-1) == "rest")
    series["exercicio"] = series["category"].apply(
        lambda c: ", ".join(dict.fromkeys(x for x in c if x != "unknown")) if isinstance(c, list) else ""
    )
    ativas = series[series["set_type"] == "active"].reset_index(drop=True)
    ativas.index += 1
    ativas["inicio"] = ativas["start_time"].dt.strftime("%H:%M:%S")
    ativas["duracao_s"] = ativas["duration"].round()
    display(ativas[["inicio", "exercicio", "repetitions", "weight", "duracao_s", "descanso_s"]].round(0))

As séries ativas marcadas sobre a frequência cardíaca:

In [ ]:
if not series.empty:
    fig = px.line(registros, x="minuto", y="heart_rate", title="FC com as séries marcadas",
                  labels={"minuto": "Minutos", "heart_rate": "FC (bpm)"})
    for _, serie in ativas.iterrows():
        x0 = (serie["start_time"] - inicio).total_seconds() / 60
        fig.add_vrect(x0=x0, x1=x0 + serie["duration"] / 60, fillcolor="#4c9be8", opacity=0.2, line_width=0)
    fig.show()